# Lab 07: Chain Composition

**Goal:** Connect multiple chains -- the output of one becomes the input of the next.

**What you'll learn:**
- How to chain two independent chains together
- Using lambda to transform data between chains
- RunnablePassthrough for passing data through unchanged
- Building complex workflows from simple pieces

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = ChatOllama(model="llama3.2:1b")

## Step 1: Two Independent Chains

Chain 1 generates a topic. Chain 2 writes about it.

In [ ]:
topic_prompt = ChatPromptTemplate.from_template(
    "Give me one specific, interesting topic about {subject}. Reply with just the topic name, nothing else."
)
topic_chain = topic_prompt | llm | StrOutputParser()

topic = topic_chain.invoke({"subject": "space exploration"})
print(f"Generated topic: {topic}")

In [ ]:
write_prompt = ChatPromptTemplate.from_template(
    "Write a 3-sentence explanation of: {topic}"
)
write_chain = write_prompt | llm | StrOutputParser()

explanation = write_chain.invoke({"topic": topic})
print(f"Explanation: {explanation}")

## Step 2: Connect Chains with Lambda

Chain 1 outputs a string, but Chain 2 expects `{"topic": string}`.
A lambda bridges the gap.

In [ ]:
full_chain = (
    topic_chain
    | (lambda topic: {"topic": topic})  # Transform: str -> dict
    | write_chain
)

result = full_chain.invoke({"subject": "artificial intelligence"})
print(f"Result: {result}")

## Step 3: Three-Stage Pipeline

- Stage 1: Generate a topic
- Stage 2: Write an explanation
- Stage 3: Simplify the explanation

In [ ]:
simplify_prompt = ChatPromptTemplate.from_template(
    "Rewrite this so a 10-year-old can understand it. Use simple words:\n\n{text}"
)
simplify_chain = simplify_prompt | llm | StrOutputParser()

three_stage = (
    topic_chain
    | (lambda topic: {"topic": topic})
    | write_chain
    | (lambda explanation: {"text": explanation})
    | simplify_chain
)

result = three_stage.invoke({"subject": "quantum computing"})
print(f"Kid-friendly explanation: {result}")

## Step 4: RunnablePassthrough -- Pass Data Through

Sometimes you need the original input alongside generated output.
`RunnablePassthrough` passes data through unchanged.

In [ ]:
# This runs two things in parallel:
#   1. Passes the question through as-is
#   2. Generates an answer via the chain
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Be concise. One sentence."),
    ("human", "{question}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()

qa_chain = RunnableParallel(
    question=RunnablePassthrough(),  # Pass input through
    answer=answer_chain,             # Generate answer
)

result = qa_chain.invoke({"question": "What is Flask?"})
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")

## TODO 1: Build a "Topic -> Quiz" Pipeline

- **Chain 1:** Takes a `{subject}` and generates a specific topic
- **Chain 2:** Takes the topic and creates a quiz question with 4 choices
- **Bonus:** Add Chain 3 that generates the answer explanation

Hint: `topic_chain` is already defined above.

In [ ]:
# quiz_prompt = ChatPromptTemplate.from_template(
#     "Create a multiple-choice quiz question about: {topic}\n"
#     "Format: Question, then A), B), C), D) options."
# )
# quiz_chain = quiz_prompt | llm | StrOutputParser()
#
# topic_to_quiz = (
#     topic_chain
#     | (lambda t: {"topic": t})
#     | quiz_chain
# )
# print(topic_to_quiz.invoke({"subject": "Python programming"}))

## TODO 2: Build a "Generate → Critique → Improve" Pipeline

A powerful pattern in AI development:

- **Chain 1:** Generate a Python function from a description
- **Chain 2:** Critique the generated code (find issues)
- **Chain 3:** Improve the code based on the critique

This is a 3-stage pipeline where each chain's output feeds the next.

In [ ]:
# generate_prompt = ChatPromptTemplate.from_template(
#     "Write a short Python function for: {task}\nReturn only the code, no explanation."
# )
# critique_prompt = ChatPromptTemplate.from_template(
#     "Review this Python code and list 1-2 issues or improvements:\n{code}"
# )
# improve_prompt = ChatPromptTemplate.from_template(
#     "Improve this code based on the feedback.\n\nCode:\n{code}\n\nFeedback:\n{critique}\n\nReturn only the improved code."
# )
#
# generate_chain = generate_prompt | llm | StrOutputParser()
# critique_chain = critique_prompt | llm | StrOutputParser()
# improve_chain = improve_prompt | llm | StrOutputParser()
#
# # Run the pipeline manually to see each stage
# task = "check if a string is a valid email address"
#
# code = generate_chain.invoke({"task": task})
# print(f"--- Generated Code ---\n{code}\n")
#
# critique = critique_chain.invoke({"code": code})
# print(f"--- Critique ---\n{critique}\n")
#
# improved = improve_chain.invoke({"code": code, "critique": critique})
# print(f"--- Improved Code ---\n{improved}")

## TODO 3: RunnableParallel -- Run Multiple Analyses

Given a text, run three analyses in parallel:
1. Sentiment (positive/negative/neutral)
2. Summary (one sentence)
3. Key topics (list of keywords)

```python
parallel_analysis = RunnableParallel(
    sentiment=sentiment_chain,
    summary=summary_chain,
    topics=topics_chain,
)
result = parallel_analysis.invoke({"text": "your text here"})
```

In [ ]:
# Your code here

## Key Takeaways

- Lambda bridges transform data between chains
- Multi-stage pipelines: `chain1 | transform | chain2 | ...`
- `RunnablePassthrough` passes data through unchanged
- `RunnableParallel` runs multiple chains simultaneously
- Complex agents are built from simple, composable chains